In [ ]:
import ee

ee.Authenticate()

In [ ]:
import ee
import os
import urllib.request
import zipfile

# ------------------------------------------------------------
# AUTENTICACIÓN
# ------------------------------------------------------------
# Ejecutar SOLO la primera vez:
# ee.Authenticate()

ee.Initialize()

# ------------------------------------------------------------
# MUNICIPIOS
# ------------------------------------------------------------
municipios = ee.FeatureCollection("FAO/GAUL/2015/level2")

# ------------------------------------------------------------
# FILTRAR PACHO
# ------------------------------------------------------------
pacho = municipios.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Colombia'),
        ee.Filter.eq('ADM1_NAME', 'Cundinamarca'),
        ee.Filter.eq('ADM2_NAME', 'Pacho')
    )
)

# ------------------------------------------------------------
# SRTM
# ------------------------------------------------------------
srtm = ee.Image("USGS/SRTMGL1_003")

# ------------------------------------------------------------
# RECORTE
# ------------------------------------------------------------
mde_pacho = srtm.clip(pacho)

# ------------------------------------------------------------
# CARPETA SALIDA
# ------------------------------------------------------------
carpeta_salida = "data_heavy/raster"

os.makedirs(carpeta_salida, exist_ok=True)

ruta_zip = os.path.join(
    carpeta_salida,
    "mde_pacho_temp.zip"
)

# ------------------------------------------------------------
# URL DESCARGA
# ------------------------------------------------------------
print("Generando URL...")

url = mde_pacho.getDownloadURL({
    'name': 'mde_pacho_srtm',
    'scale': 30,
    'crs': 'EPSG:4326',
    'region': pacho.geometry(),
    'format': 'GEO_TIFF'
})

print("Descargando...")

# ------------------------------------------------------------
# DESCARGA
# ------------------------------------------------------------
urllib.request.urlretrieve(url, ruta_zip)

print("Extrayendo GeoTIFF...")

# ------------------------------------------------------------
# EXTRAER ZIP
# ------------------------------------------------------------
with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:

    nombre_original = zip_ref.namelist()[0]

    zip_ref.extractall(carpeta_salida)

# ------------------------------------------------------------
# RENOMBRAR
# ------------------------------------------------------------
ruta_original = os.path.join(
    carpeta_salida,
    nombre_original
)

ruta_final = os.path.join(
    carpeta_salida,
    "mde_pacho_srtm_30m.tif"
)

if os.path.exists(ruta_original):

    os.rename(ruta_original, ruta_final)

# ------------------------------------------------------------
# LIMPIEZA
# ------------------------------------------------------------
os.remove(ruta_zip)

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------
print("========================================")
print("DESCARGA COMPLETADA")
print("========================================")
print(f"Archivo:\n{ruta_final}")
print("========================================")